In [134]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import random

In [135]:
data = pd.read_csv("D:\\Ujjawal's Data\\Power BI Desktop\\Nike Analysis\\Nike_Sales_Uncleaned.csv")

In [136]:
data.head

<bound method NDFrame.head of       Order_ID Gender_Category Product_Line      Product_Name Size  \
0         2000            Kids     Training       SuperRep Go    M   
1         2001           Women       Soccer     Tiempo Legend    M   
2         2002           Women       Soccer       Premier III    M   
3         2003            Kids    Lifestyle        Blazer Mid    L   
4         2004            Kids      Running    React Infinity   XL   
...        ...             ...          ...               ...  ...   
2495      4495            Kids   Basketball     Kyrie Flytrap   XL   
2496      4496             Men   Basketball     Kyrie Flytrap    L   
2497      4497             Men       Soccer     Tiempo Legend    7   
2498      4498           Women     Training  ZoomX Invincible  NaN   
2499      4499           Women      Running          Air Zoom    M   

      Units_Sold      MRP  Discount_Applied  Revenue  Order_Date  \
0            NaN      NaN              0.47      0.0  2024-03

In [137]:
data.columns = data.columns.str.strip()

In [138]:
data['Order_ID'] = data['Order_ID'].apply(lambda x: f"ORD-{int(x):04d}")

In [139]:
data.isnull().sum()

Order_ID               0
Gender_Category        0
Product_Line           0
Product_Name           0
Size                 510
Units_Sold          1235
MRP                 1254
Discount_Applied    1668
Revenue                0
Order_Date           616
Sales_Channel          0
Region                 0
Profit                 0
dtype: int64

In [140]:
data = data.drop_duplicates()

In [141]:
data.head

<bound method NDFrame.head of       Order_ID Gender_Category Product_Line      Product_Name Size  \
0     ORD-2000            Kids     Training       SuperRep Go    M   
1     ORD-2001           Women       Soccer     Tiempo Legend    M   
2     ORD-2002           Women       Soccer       Premier III    M   
3     ORD-2003            Kids    Lifestyle        Blazer Mid    L   
4     ORD-2004            Kids      Running    React Infinity   XL   
...        ...             ...          ...               ...  ...   
2495  ORD-4495            Kids   Basketball     Kyrie Flytrap   XL   
2496  ORD-4496             Men   Basketball     Kyrie Flytrap    L   
2497  ORD-4497             Men       Soccer     Tiempo Legend    7   
2498  ORD-4498           Women     Training  ZoomX Invincible  NaN   
2499  ORD-4499           Women      Running          Air Zoom    M   

      Units_Sold      MRP  Discount_Applied  Revenue  Order_Date  \
0            NaN      NaN              0.47      0.0  2024-03

In [142]:
data['Order_Date'] = pd.to_datetime(data['Order_Date'], dayfirst=True, errors='coerce')

In [143]:
data['Year'] = data['Order_Date'].dt.year
data['Month'] = data['Order_Date'].dt.strftime('%b')
data['Day'] = data['Order_Date'].dt.day

In [144]:
data['Size'] = pd.to_numeric(data['Size'], errors='coerce')

In [145]:
def convert_size(x):
    if x == "M":
        return 8.5
    elif x == "L":
        return 9.5
    elif x == "XL":
        return 11
    else:
        return x
    
data['Size'] = data['Size'].apply(convert_size)

In [146]:
conditions = [
    (data['Size'] >= 6) & (data['Size'] <= 7),
    (data['Size'] >= 8) & (data['Size'] <= 9),
    (data['Size'] >= 9) & (data['Size'] <= 10),
    (data['Size'] >= 11) & (data['Size'] <= 12)
]

choices = ['Small', 'Medium', 'Large', 'Extra Large']

data['Size_Category'] = np.select(conditions, choices, default='Other')

In [147]:

data['Product_Status'] = np.where( data['Units_Sold'] < 0,'Returned','Delivered')

In [148]:
mask = data['Size'].isna()
data.loc[mask, 'Size'] = np.random.randint(6, 13, size=mask.sum())

In [149]:

low = data['MRP'].mean()
high = data['MRP'].median()
mask = data['MRP'].isna()
data.loc[mask, 'MRP'] = np.round( np.random.uniform(low, high, size=mask.sum()), 2)


In [150]:
data['Discount_Applied'] = data['Discount_Applied'].fillna(0)
data['Units_Sold'] = data['Units_Sold'].fillna(0)
data['Revenue'] = np.where( data['Product_Status'] == 'Delivered', data['Units_Sold'] * data['MRP'] * (1 - data['Discount_Applied']), 0)


In [157]:
data['Profit'] = np.where(
    data['Revenue'] > 0,
    0.35 * data['Revenue'],
    data['Revenue']
)

In [158]:
data.to_excel("D:\\Ujjawal's Data\\Power BI Desktop\\Nike Analysis\\Cleaned_output_final.xlsx", index=False)